# Running SuSiE Fine-mapping Pipeline with Nextflow

This notebook demonstrates how to run the Nextflow fine-mapping pipeline and visualize results.

**Note**: For production/long-running pipelines, consider running from terminal with `nohup`.

## 1. Setup and Configuration

In [7]:
import os
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# CRITICAL: Activate gwas_tutorial environment to get correct Python packages
# This ensures gwaslab, rpy2, susieR are available with compatible versions
print("Activating gwas_tutorial conda environment...")
os.environ['CONDA_DEFAULT_ENV'] = 'gwas_tutorial'
os.environ['PATH'] = '/home/sagemaker-user/.conda/envs/gwas_tutorial/bin:' + os.environ.get('PATH', '')

# Set working directory
os.chdir('/home/sagemaker-user/nextflow')
print(f"Current directory: {os.getcwd()}")
print(f"Using conda env: {os.environ.get('CONDA_DEFAULT_ENV', 'unknown')}")

Activating gwas_tutorial conda environment...
Current directory: /home/sagemaker-user/nextflow
Using conda env: gwas_tutorial


## 2. Check Nextflow Installation

In [8]:
# Check Nextflow version
!nextflow -version


      N E X T F L O W
      version 26.04.4 build 12445
      created 17-06-2026 16:30 UTC 
      cite doi:10.1038/nbt.3820
      http://nextflow.io



## 3. Check Input Files Exist

In [9]:
# Define input paths
sumstats_file = "/home/sagemaker-user/GWASTutorial/12_fine_mapping/1kgeas.B1.glm.firth"
genotype_prefix = "/home/sagemaker-user/GWASTutorial/01_Dataset/1KG.EAS.auto.snp.norm.nodup.split.rare002.common015.missing"

# Check if files exist
print("Checking input files...")
print(f"Summary stats: {os.path.exists(sumstats_file)}")
print(f"Genotype .bed: {os.path.exists(genotype_prefix + '.bed')}")
print(f"Genotype .bim: {os.path.exists(genotype_prefix + '.bim')}")
print(f"Genotype .fam: {os.path.exists(genotype_prefix + '.fam')}")

# Quick peek at summary stats
if os.path.exists(sumstats_file):
    df_preview = pd.read_csv(sumstats_file, sep="\t", nrows=5)
    print(f"\nSummary stats preview (first 5 rows):")
    display(df_preview)

Checking input files...
Summary stats: True
Genotype .bed: True
Genotype .bim: True
Genotype .fam: True

Summary stats preview (first 5 rows):


,#CHROM,POS,ID,REF,ALT,PROVISIONAL_REF?,A1,OMITTED,A1_FREQ,TEST,OBS_CT,OR,LOG(OR)_SE,Z_STAT,P,ERRCODE
0,1,15774,1:15774:G:A,G,A,Y,A,G,0.028283,ADD,495,NaN,NaN,NaN,NaN,FIRTH_CONVERGE_FAIL
1,1,15777,1:15777:A:G,A,G,Y,G,A,0.073737,ADD,495,NaN,NaN,NaN,NaN,FIRTH_CONVERGE_FAIL
2,1,57292,1:57292:C:T,C,T,Y,T,C,0.104675,ADD,492,NaN,NaN,NaN,NaN,FIRTH_CONVERGE_FAIL
3,1,77874,1:77874:G:A,G,A,Y,A,G,0.019153,ADD,496,1.12228,0.46275,0.249299,0.80313,.
4,1,87360,1:87360:C:T,C,T,Y,T,C,0.023139,ADD,497,NaN,NaN,NaN,NaN,FIRTH_CONVERGE_FAIL


## 4. Configure Pipeline Parameters

In [10]:
# Define parameters
params = {
    "sumstats": sumstats_file,
    "genotype_prefix": genotype_prefix,
    "outdir": "finemapping_results",
    "sig_threshold": 5e-8,
    "window_size": 500000,  # 500kb
    "coverage": 0.95,
    "min_abs_corr": 0.5,
    "L": 1
}

print("Pipeline Parameters:")
print("=" * 60)
for key, value in params.items():
    print(f"{key:20s}: {value}")
print("=" * 60)

Pipeline Parameters:
sumstats            : /home/sagemaker-user/GWASTutorial/12_fine_mapping/1kgeas.B1.glm.firth
genotype_prefix     : /home/sagemaker-user/GWASTutorial/01_Dataset/1KG.EAS.auto.snp.norm.nodup.split.rare002.common015.missing
outdir              : finemapping_results
sig_threshold       : 5e-08
window_size         : 500000
coverage            : 0.95
min_abs_corr        : 0.5
L                   : 1


## 5. Run Nextflow Pipeline

**Important**: We run Nextflow from the `gwas_tutorial` environment (set above) WITHOUT `-profile conda`.
- The `gwas_tutorial` environment has all required packages (gwaslab, rpy2, susieR) with compatible versions
- When Nextflow runs from this environment, all Python processes inherit it automatically
- The `nextflow-env` environment has an incompatible gwaslab version that causes errors

**Note**: This cell may take several minutes to complete depending on:
- Number of significant loci
- Size of genomic windows
- Computing resources

You'll see real-time progress as the pipeline runs.

In [16]:
# Build the command - NO -profile conda flag!
# We're already in gwas_tutorial environment (see cell 2)
cmd = [
    "nextflow", "run", "finemapping_susie.nf",
    # NOTE: NO -profile conda here - we run from gwas_tutorial environment directly
    "--sumstats", params["sumstats"],
    "--genotype_prefix", params["genotype_prefix"],
    "--outdir", params["outdir"],
    "--sig_threshold", str(params["sig_threshold"]),
    "--window_size", str(params["window_size"]),
    "--coverage", str(params["coverage"]),
    "--min_abs_corr", str(params["min_abs_corr"]),
    "--L", str(params["L"])
]

print("Running command:")
print(" ".join(cmd))
print("\n" + "="*60)
print("Pipeline execution started...")
print("="*60 + "\n")

# Run the pipeline
result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print("\n" + "="*60)
    print("✅ Pipeline completed successfully!")
    print("="*60)
else:
    print("\n" + "="*60)
    print("❌ Pipeline failed with errors")
    print("="*60)

Running command:
nextflow run finemapping_susie.nf --sumstats /home/sagemaker-user/GWASTutorial/12_fine_mapping/1kgeas.B1.glm.firth --genotype_prefix /home/sagemaker-user/GWASTutorial/01_Dataset/1KG.EAS.auto.snp.norm.nodup.split.rare002.common015.missing --outdir finemapping_results --sig_threshold 5e-08 --window_size 500000 --coverage 0.95 --min_abs_corr 0.5 --L 1

Pipeline execution started...


 N E X T F L O W   ~  version 26.04.4

Launching `finemapping_susie.nf` [agitated_koch] revision: cf08896a9a


    Fine-mapping with SuSiE-RSS - Nextflow Pipeline
    Summary statistics  : /home/sagemaker-user/GWASTutorial/12_fine_mapping/1kgeas.B1.glm.firth
    Genotype prefix     : /home/sagemaker-user/GWASTutorial/01_Dataset/1KG.EAS.auto.snp.norm.nodup.split.rare002.common015.missing
    Output directory    : finemapping_results
    Significance level  : 5e-08
    Window size         : 500000 bp
    Credible set cover  : 0.95
    Min correlation     : 0.5
    Max causal (L)      : 1
    Re

## 6. Check Output Files

In [ ]:
# List output directory structure
output_dir = Path(params["outdir"])

if output_dir.exists():
    print("Output directory structure:")
    print("="*60)
    
    for dirpath, dirnames, filenames in os.walk(output_dir):
        level = dirpath.replace(str(output_dir), '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(dirpath)}/")
        subindent = ' ' * 2 * (level + 1)
        for filename in filenames:
            print(f"{subindent}{filename}")
else:
    print(f"Output directory {output_dir} does not exist yet.")

## 7. Load and Examine Lead Variants

In [ ]:
# Load lead variants
lead_file = Path(params["outdir"]) / "leads" / "lead_variants.tsv"

if lead_file.exists():
    leads = pd.read_csv(lead_file, sep="\t")
    print(f"Number of genome-wide significant loci: {len(leads)}")
    print("\nLead variants:")
    display(leads[["SNPID", "CHR", "POS", "EA", "NEA", "P", "OR", "Z"]])
else:
    print(f"Lead variants file not found: {lead_file}")

## 8. Display Manhattan Plot

In [ ]:
from IPython.display import Image, display

manhattan_file = Path(params["outdir"]) / "leads" / "manhattan.png"

if manhattan_file.exists():
    print("Manhattan Plot:")
    display(Image(filename=str(manhattan_file)))
else:
    print(f"Manhattan plot not found: {manhattan_file}")

## 9. Load Fine-mapping Results

In [ ]:
# Load all credible sets
credible_file = Path(params["outdir"]) / "summary" / "all_credible_sets.tsv"

if credible_file.exists():
    credible_sets = pd.read_csv(credible_file, sep="\t")
    
    if len(credible_sets) > 0:
        print(f"Total variants in credible sets: {len(credible_sets)}")
        print(f"Number of unique credible sets: {credible_sets['cs'].nunique()}")
        
        print("\nTop 10 variants by PIP (Posterior Inclusion Probability):")
        top_variants = credible_sets.nlargest(10, "pip")[["SNPID", "CHR", "POS", "cs", "pip", "P", "BETA"]]
        display(top_variants)
        
        # Summary by credible set
        print("\nSummary by credible set:")
        cs_summary = credible_sets.groupby("cs").agg({
            "SNPID": "count",
            "pip": ["max", "sum"]
        }).round(4)
        cs_summary.columns = ["n_variants", "max_pip", "total_pip"]
        display(cs_summary)
    else:
        print("No credible sets were identified.")
else:
    print(f"Credible sets file not found: {credible_file}")

## 10. Display Regional Plots

In [ ]:
# Find all regional plot files
finemapping_dir = Path(params["outdir"]) / "finemapping"

if finemapping_dir.exists():
    plot_files = sorted(finemapping_dir.glob("plot_*.png"))
    
    if plot_files:
        print(f"Found {len(plot_files)} regional plot(s)\n")
        
        for plot_file in plot_files:
            print(f"\nRegional Plot: {plot_file.name}")
            print("="*60)
            display(Image(filename=str(plot_file)))
    else:
        print("No regional plots found.")
else:
    print(f"Fine-mapping directory not found: {finemapping_dir}")

## 11. View Summary Report

In [ ]:
# Display summary report
summary_file = Path(params["outdir"]) / "summary" / "summary_report.txt"

if summary_file.exists():
    print("Pipeline Summary Report:")
    print("="*60)
    with open(summary_file, "r") as f:
        print(f.read())
else:
    print(f"Summary report not found: {summary_file}")

## 12. View Execution Reports

In [ ]:
# List report files
report_dir = Path(params["outdir"]) / "reports"

if report_dir.exists():
    print("Available execution reports:")
    print("="*60)
    
    for report_file in report_dir.iterdir():
        if report_file.is_file():
            print(f"  {report_file.name}")
    
    print("\nYou can view HTML reports by downloading them or using a file browser.")
    print(f"Location: {report_dir}")
else:
    print(f"Reports directory not found: {report_dir}")

## 13. Advanced: Compare with Original Notebook Results

If you have results from the original notebook, compare them here.

In [ ]:
# Load original notebook results if available
notebook_results = "/home/sagemaker-user/GWASTutorial/12_fine_mapping/sig_locus.tsv"

if os.path.exists(notebook_results):
    original = pd.read_csv(notebook_results, sep="\t")
    print(f"Original notebook had {len(original)} variants in the locus")
    
    if credible_file.exists():
        pipeline_results = pd.read_csv(credible_file, sep="\t")
        
        # Compare if same locus
        if len(pipeline_results) > 0:
            print(f"Pipeline found {len(pipeline_results)} variants in credible sets")
            
            # Find common variants
            common = set(original["SNPID"]) & set(pipeline_results["SNPID"])
            print(f"Common variants: {len(common)}")
else:
    print("Original notebook results not found for comparison.")

## 14. Clean Up (Optional)

Uncomment and run this cell to clean up work directory and results.

In [ ]:
# # Clean up work directory (frees disk space)
# !rm -rf work/
# print("Work directory cleaned")

# # Remove results (be careful!)
# # !rm -rf finemapping_results/
# # print("Results removed")

## 📝 Notes

### Why We Run from gwas_tutorial Environment:
The pipeline requires specific Python packages:
- **gwaslab**: Working version in gwas_tutorial, broken in nextflow-env
- **rpy2 + susieR**: For statistical fine-mapping
- **plink**: For LD calculation

**Solution**: Activate gwas_tutorial, run Nextflow WITHOUT `-profile conda`, and all processes inherit the correct environment.

### For Production Use (from terminal):
```bash
cd /home/sagemaker-user/nextflow

# Activate the correct environment
conda activate gwas_tutorial

# Install nextflow in this environment if needed
conda install -c bioconda nextflow

# Run pipeline
nohup nextflow run finemapping_susie.nf \
  --sumstats /path/to/sumstats \
  --genotype_prefix /path/to/genotypes \
  --outdir results \
  > nextflow.log 2>&1 &

# Monitor
tail -f nextflow.log
```

### Resume Failed Pipeline:
```bash
nextflow run finemapping_susie.nf -resume [other params...]
```

### Check Pipeline Status:
```bash
# View work directory
ls -lh work/

# View logs
cat .nextflow.log
```

### Debugging Tips:
```bash
# Check which Python is being used in failed tasks
find work -name ".command.err" -type f | xargs ls -t | head -1 | xargs cat

# Look at the full command that was run
find work -name ".command.run" -type f | head -1 | xargs cat
```